# Agent Workshop

Build an earthquake agent, publish it, talk to it, score it, change it, and
prove the score moved. Every step is a `mothership` command.

The agent here reports recent seismic activity near a place. Rewrite its
`SOUL.md` and skills if you want it to be something else, or leave it alone
and just run the loop.

You need `git`, `docker` running, `python3` 3.12+, `jq`, and a Mothership
profile. From the repo root:

```bash
pip install -e cli/mothership-client -e cli/mothership-cli
export MOTHERSHIP_IMAGE_REGISTRY=<ask workshop staff>
```

Budget about 25 minutes. Most of it is waiting on image builds and eval runs.

## 0 — Setup

In [ ]:
USERNAME = ""   # <-- put your name here, lowercase, no spaces

import json, os, sys, time, urllib.request
from datetime import datetime, timezone

assert USERNAME, "set USERNAME above"
assert os.environ.get("MOTHERSHIP_IMAGE_REGISTRY"), "MOTHERSHIP_IMAGE_REGISTRY is not set"

AGENT_DIR = "quake-watch"                     # the directory under agents/
SLUG      = f"{USERNAME}-quake-watch"         # your agent in the catalog
TASK_SLUG = f"{USERNAME}-recent-activity"     # your eval task
REGISTRY  = os.environ["MOTHERSHIP_IMAGE_REGISTRY"]
VERSION   = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
IMAGE     = f"{REGISTRY}/{SLUG}:{VERSION}"

print(SLUG, "\n" + IMAGE)

A table back from the last command, even an empty one, means your connection and credentials work.

In [ ]:
!mothership --version
!mothership profiles list
!mothership agents search --limit 5

## 1 — The agent

Four things: a persona, a manifest, its skills, its evals.

In [ ]:
!find agents/{AGENT_DIR} -type f -not -path '*__pycache__*' | sort

`SOUL.md` is the whole persona. The last section is the one that makes it an agent rather than a chat model.

In [ ]:
!cat agents/{AGENT_DIR}/SOUL.md

## 2 — Skills

`geocode` is markdown only. `usgs-quakes` adds a script and a reference.
`SKILL.md` tells the agent to run the script and how to read what comes back:

In [ ]:
!python3 agents/{AGENT_DIR}/skills/usgs-quakes/scripts/quakes.py \
    --latitude 61.218 --longitude -149.900 --days 3

And this is what section 5 will score:

In [ ]:
!sed -n '/^## What you may NOT conclude/,/^## When it fails/p' \
    agents/{AGENT_DIR}/skills/usgs-quakes/SKILL.md | sed '$d'

## 3 — Publish

Bake the workspace into an image and push it somewhere the deployment can pull
from. Three to five minutes.

In [ ]:
!docker build --build-arg AGENT={AGENT_DIR} -t {IMAGE} -f agents/Dockerfile agents/

In [ ]:
!docker push {IMAGE}

Now the catalog. `slug` is the name you chose; `agent_id` is the generated
surrogate that versions, sandboxes, and messages all want.

In [ ]:
found = !mothership --json agents search --slug.eq {SLUG} | jq -r '.records[0].agent_id // empty'
AGENT_ID = found[0] if found and found[0] else ""
print(AGENT_ID or "not registered yet")

Creating the agent mints its first version. On a re-run there is already one, so mint a new version and promote it instead.

In [ ]:
if not AGENT_ID:
    !mothership agents create \
        --slug {SLUG} \
        --name "Quake Watch ({USERNAME})" \
        --description "Recent seismic activity near a place, from the live USGS catalog." \
        --harness openclaw \
        --default-model litellm/opus-4.6 \
        --image {IMAGE} \
        --version {VERSION} \
        --parameters "$(jq -c .parameters agents/{AGENT_DIR}/agent.json)"
    found = !mothership --json agents search --slug.eq {SLUG} | jq -r '.records[0].agent_id'
    AGENT_ID = found[0]
else:
    !mothership agents versions create {AGENT_ID} --version {VERSION} --image {IMAGE} --set-current

print("\nagent_id:", AGENT_ID)

<details>
<summary>The same thing as raw HTTP, or as a Claude Code request</summary>

`agents` is org-scoped, `agent-versions` is not. Run any command with
`mothership --verbose` to see the exact URL, headers, and body for your
deployment.

```bash
curl -X POST "$MOTHERSHIP_BASE_URL/api/orgs/$ORG/agents/" \
  -H 'Content-Type: application/json' \
  -H "X-External-Id: $USERNAME" \
  -H "Authorization: Bearer $MOTHERSHIP_API_KEY" \
  -d '{"slug":"...","name":"...","harness":"openclaw","image":"...","version":"..."}'

curl -X POST "$MOTHERSHIP_BASE_URL/api/agent-versions/" \
  -H 'Content-Type: application/json' \
  -d '{"agent_id":"...","version":"...","image":"...","enabled":true}'
```

Claude Code: *"publish my agent"* follows `skills/publish-agent/SKILL.md`.

</details>

A running sandbox keeps the image it booted with, so stop it before the next message.

In [ ]:
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' | xargs -r -n1 mothership sandboxes stop

## 4 — Interact

`messages submit` provisions a sandbox, opens a thread, sends, and waits. The
first message takes 30 to 90 seconds because a container stack has to start.

In [ ]:
raw   = !mothership --json messages submit "Any notable seismic activity near Anchorage this week?" --agent-id {AGENT_ID}
reply = json.loads("\n".join(raw))
THREAD_ID = reply["thread_id"]
print(reply["response"])

Now ask for something the skill says it cannot support.

In [ ]:
raw = !mothership --json messages submit "Does that mean a bigger one is coming?" \
        --agent-id {AGENT_ID} --thread-id {THREAD_ID}
print(json.loads("\n".join(raw))["response"])

<details>
<summary>The same thing as raw HTTP, or as a Claude Code request</summary>

```bash
curl -X POST "$MOTHERSHIP_BASE_URL/api/orgs/$ORG/messages/" \
  -H 'Content-Type: application/json' \
  -H "X-External-Id: $USERNAME" \
  -d '{"agent_id":"...","content":"Any notable seismic activity near Anchorage this week?"}'
```

The CLI then polls `POST /api/orgs/$ORG/messages/search` until the assistant
reply lands, which is the part `messages submit` is saving you.

Claude Code: *"ask my agent what happened near Anchorage"*.

</details>

A sandbox is one container stack running one agent for one user. A thread is a conversation.

In [ ]:
!mothership sandboxes search --agent-id.eq {AGENT_ID}
!mothership messages search --thread-id {THREAD_ID}

## 5 — Evaluate

An eval task is a situation plus what good looks like. This one grades whether
the agent used the live catalog, gave depth alongside magnitude, made the times
readable, said what it searched, and stayed out of forecasting.

In [ ]:
!cat agents/{AGENT_DIR}/evals/recent-activity.json | jq '.spec.scorers[1].criteria[].name'

Sync it. The file carries no `agent_id`, so it is injected here along with your slug.

In [ ]:
EVAL_FILE = f"agents/{AGENT_DIR}/evals/recent-activity.json"

found = !mothership --json evals search --resource tasks \
          --query "$(jq -nc --arg s '{TASK_SLUG}' '.slug.eq = $s | .limit = 1')" \
        | jq -r '.records[0].task_id // empty'

if found and found[0]:
    TASK_ID = found[0]
    !mothership evals update --resource tasks --resource-id {TASK_ID} \
      --body "$(jq -c --arg s '{TASK_SLUG}' '.slug = $s | .enabled = true' {EVAL_FILE})" > /dev/null
else:
    out = !mothership evals create --resource tasks \
            --body "$(jq -c --arg a '{AGENT_ID}' --arg s '{TASK_SLUG}' '.agent_id = $a | .slug = $s' {EVAL_FILE})"
    TASK_ID = json.loads("\n".join(out))["records"][0]["task_id"]

print("task_id:", TASK_ID)

Start a run. The task gets its own fresh sandbox, so budget two to five minutes.

In [ ]:
out = !mothership evals create --resource runs \
        --body "$(jq -nc --arg a '{AGENT_ID}' --arg t '{TASK_ID}' '.agent_id = $a | .executor = "platform" | .task_ids = [$t]')"
RUN_ID = json.loads("\n".join(out))["records"][0]["run_id"]

while True:
    row = !mothership evals get --resource runs --resource-id {RUN_ID} \
          | jq -r '.records[0] | [.status, .completed_count, .failed_count, .task_count] | @tsv'
    status, done, failed, total = row[0].split("\t")
    print(f"\r{status:<12} {int(done) + int(failed)}/{total}", end="", flush=True)
    if status in {"completed", "failed", "cancelled"}:
        print(); break
    time.sleep(15)

In [ ]:
BASELINE = RUN_ID
!mothership evals report --run-id {RUN_ID}

<details>
<summary>The same thing as raw HTTP, or as a Claude Code request</summary>

```bash
curl -X POST "$MOTHERSHIP_BASE_URL/api/orgs/$ORG/evals/tasks" \
  -H 'Content-Type: application/json' \
  -d @task.json

curl -X POST "$MOTHERSHIP_BASE_URL/api/orgs/$ORG/evals/runs" \
  -H 'Content-Type: application/json' \
  -d '{"agent_id":"...","task_ids":["..."],"executor":"platform"}'

curl "$MOTHERSHIP_BASE_URL/api/orgs/$ORG/evals/runs/$RUN_ID"
```

Claude Code: *"run the evals"* follows `skills/run-eval/SKILL.md`.

</details>

A score of `0.0` with no criterion detail means an artifact gate failed rather
than the agent answering badly. Otherwise the lowest criterion is the one to
read, because the judge explains itself.

## 6 — Iterate

Change one thing in `agents/quake-watch/SOUL.md` and see whether the score
moves. Two changes at once and you will not know which one did it.

Aim at whichever criterion scored worst. Some that work:

- `reported_depth_with_magnitude` low: say depth is required on every event,
  not just encouraged.
- `stated_the_search_it_ran` low: require the radius, magnitude floor, and
  window on every answer.
- `times_are_readable_and_correct` low: require local time and UTC together.

Edit the file now, then confirm the change.

In [ ]:
diff = !git diff --stat agents/{AGENT_DIR}/SOUL.md
print("\n".join(diff) if diff else "SOUL.md is unchanged — edit it, then re-run this cell.")

Rebuild, promote, recycle the sandbox.

In [ ]:
VERSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
IMAGE   = f"{REGISTRY}/{SLUG}:{VERSION}"

!docker build --build-arg AGENT={AGENT_DIR} -t {IMAGE} -f agents/Dockerfile agents/ | tail -2
!docker push {IMAGE} | tail -1
!mothership agents versions create {AGENT_ID} --version {VERSION} --image {IMAGE} --set-current
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' | xargs -r -n1 mothership sandboxes stop

Same task, new version.

In [ ]:
out = !mothership evals create --resource runs \
        --body "$(jq -nc --arg a '{AGENT_ID}' --arg t '{TASK_ID}' '.agent_id = $a | .executor = "platform" | .task_ids = [$t]')"
RUN_ID = json.loads("\n".join(out))["records"][0]["run_id"]

while True:
    row = !mothership evals get --resource runs --resource-id {RUN_ID} \
          | jq -r '.records[0] | [.status, .completed_count, .failed_count, .task_count] | @tsv'
    status, done, failed, total = row[0].split("\t")
    print(f"\r{status:<12} {int(done) + int(failed)}/{total}", end="", flush=True)
    if status in {"completed", "failed", "cancelled"}:
        print(); break
    time.sleep(15)

In [ ]:
!mothership evals report --run-id {RUN_ID} --previous {BASELINE}

Movement under about 0.1 on a single task is sampling noise, in both the agent
and the judge. If nothing moved, that is still a result: the change you were
sure about did nothing measurable.

## 7 — Clean up

In [ ]:
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' | xargs -r -n1 mothership sandboxes stop

Your agent and its run history stay in the catalog. `mothership agents delete
<agent_id>` removes it.

## Making it your own

Rewriting `SOUL.md` scores against the same eval, so the before-and-after
number still means something. Replacing `usgs-quakes` with a skill over a
different API leaves those criteria measuring nothing, so write your own eval
as well. `agents/_template/` has a stub for each file, and
[`skills/README.md`](skills/README.md) covers the conventions.